# 📚 StudyBuddy — Adaptive, Agentic Study Assistant

**Applied Generative AI — Final Project demo notebook.**

StudyBuddy turns any study material (PDF or pasted notes) into:
- extracted **key concepts**,
- **grounded quizzes** (multiple-choice / true-false / short-answer × recall / application / analysis),
- **graded answers** with explanations, tracked per concept,
- an **adaptive weakness round** that re-drills exactly what you got wrong (and escalates what you mastered),
- a **tool-calling Tutor** that answers open-ended questions (RAG) and drives the app.

It uses **≥3 LLMs with intelligent per-task routing** through one OpenAI-compatible gateway, orchestrated by **LangGraph**, with a **LangChain tool-calling agent + memory** on top.

> **The demo moment:** ingest biology material, answer the questions *wrong*, then watch the next round auto-re-drill exactly those concepts at higher difficulty — then ask the tutor a free-form question.

_Educational aid only — generated concepts, questions, grades, and tutor answers can contain errors._

## 1. Setup

In **Colab**, clone the repo and install dependencies. (Locally, you can skip the clone and just `pip install -r requirements.txt` from the project root.)

In [ ]:
import os

# In Colab: clone the project (replace with your repo URL) and enter it.
if not os.path.isdir('studybuddy'):
    !git clone https://github.com/<your-org>/StudyBuddy.git _studybuddy_repo
    %cd _studybuddy_repo

!pip install -q -r requirements.txt

## 2. Configure models & key

Set your gateway key and per-task models. With an **OpenAI** key, leave `OPENAI_BASE_URL` blank and use OpenAI model names. With a **gateway** (OpenRouter / LiteLLM), set the base URL and use slugs like `anthropic/claude-3.5-sonnet` to route genuinely different LLMs per agent.

In [ ]:
import os
from getpass import getpass

os.environ['OPENAI_API_KEY'] = getpass('OPENAI_API_KEY: ')
os.environ.setdefault('OPENAI_BASE_URL', '')   # e.g. https://openrouter.ai/api/v1

# Per-task routing (≥3 distinct models satisfies the rubric).
os.environ.setdefault('EXTRACTION_MODEL', 'gpt-4o-mini')
os.environ.setdefault('QUIZ_MODEL', 'gpt-4o')
os.environ.setdefault('EVAL_MODEL', 'gpt-4o-mini')
os.environ.setdefault('ADAPTIVE_MODEL', 'gpt-4o-mini')
os.environ.setdefault('TUTOR_MODEL', 'gpt-4o')   # must support tool calling
print('Configured. Models:', {k: os.environ[k] for k in ['EXTRACTION_MODEL','QUIZ_MODEL','EVAL_MODEL','ADAPTIVE_MODEL','TUTOR_MODEL']})

## 3. Ingest material → concepts → round-1 quiz

`ingest_material` runs the LangGraph pipeline: chunk → embed into Chroma (local embeddings) → extract concepts → generate a grounded quiz.

In [ ]:
from studybuddy.graph import ingest_material, submit_answers, next_round
from studybuddy import store
from studybuddy.tracker import concept_scores

SESSION = 'demo'
MATERIAL = '''1. Cells
Cells are the basic structural and functional units of all living organisms.
2. Mitochondria
Mitochondria are the powerhouse of the cell and produce ATP through cellular respiration.
3. Photosynthesis
Photosynthesis converts light energy into chemical energy in the chloroplasts of plants.
'''

state = ingest_material(SESSION, MATERIAL, 'biology')
print('Concepts:', [c['name'] for c in store.get_concepts(SESSION)])
print('Round-1 questions:', len(state['quiz']))
for q in state['quiz']:
    print(f" - [{q['question_type']}/{q['difficulty']}] {q['prompt']}")

## 4. Answer (some) wrong → adaptive weakness round

We answer the **first** concept correctly and everything else incorrectly. The Adaptive Agent reads the per-concept scores and plans the next round: **re-drill** the weak concepts and **escalate** the mastered one.

In [ ]:
concepts = store.get_concepts(SESSION)
quiz = store.get_latest_quiz(SESSION)
strong = concepts[0]['concept_id']
answers = {q['question_id']: (q['answer'] if q['concept_id'] == strong else 'wrong') for q in quiz}

results = submit_answers(SESSION, answers)
print('Score:', sum(r['correct'] for r in results), '/', len(results))
print('Per-concept:', [(s['concept_id'], f"{s['accuracy']:.0%}") for s in concept_scores(SESSION)])

nr = next_round(SESSION)
plan = nr['adaptive_plan']
print('\nAdaptive plan:')
print('  re-drill:', plan['redrill_concepts'])
print('  difficulty_by_concept:', plan['difficulty_by_concept'])
print('  weakness round targets:', sorted({q['concept_id'] for q in nr['quiz']}))
print('  difficulties in round:', sorted({q['difficulty'] for q in nr['quiz']}))

## 5. Talk to the Tutor (tool-calling + RAG)

The Tutor decides which tool to call: `search_material` for open-ended Q&A (cited), `quiz_me` to start a round, `get_weak_concepts` to report struggles, `explain_concept`, `list_concepts`.

In [ ]:
from studybuddy.agents.tutor import chat

r = chat(SESSION, 'What do mitochondria produce?')
print('Q: What do mitochondria produce?')
print('A:', r['answer'])
print('Sources:', r['sources'])

r2 = chat(SESSION, 'What am I weak on? Then quiz me on it.')
print('\nA:', r2['answer'])
print('Actions:', r2['actions'])

## 5b. New features — flexible quizzes, flashcards, summary, export

The expanded toolkit (Phases 9-17): regenerate a quiz with different questions, make flashcards, generate a grounded cheat-sheet, and export a Markdown study pack.

In [ ]:
from studybuddy.graph import regenerate_quiz, make_flashcards
from studybuddy.agents.summarizer import make_summary
from studybuddy.export import export_markdown

fresh = regenerate_quiz(SESSION, num_questions=3)
print('Regenerated quiz:', len(fresh), 'new questions')

cards = make_flashcards(SESSION)
print('Flashcards:', [(c['front'], c['back']) for c in cards[:2]])

print('Cheat sheet:', make_summary(SESSION)[:300])
print('Exported to:', export_markdown(SESSION))

## 6. Launch the interactive app

The Gradio UI ties it together: **Upload/Paste · Quiz · Dashboard (mastery map) · Tutor**. In Colab, `share=True` gives a public link.

In [ ]:
from studybuddy.ui import build_app

build_app().launch(share=True)